# 📄 Notebook Overview: Selecting Top Malicious AI Incidents

This notebook processes and analyzes AI-related incident data to identify the **most impactful incidents** in the "Malicious Use & Security" category. The workflow is as follows:

1. **Load Data**
   - Reads `llm_classified_incidents.csv`, which contains incidents categorized by harm type.
   - Loads `unified_incidents.json`, which contains the full details and AI classifications of each incident.

2. **Filter for Relevant Incidents**
   - Only incidents labeled **"Malicious Use & Security"** are considered for further analysis.

3. **Match and Extract Classification Data**
   - Each selected incident is matched to the full JSON record.
   - Key fields are extracted, including:
     - `entities` (users or groups impacted)
     - `AI_system` (whether AI was involved)
     - `AI_task` (AI functionality, e.g., content moderation)
     - `AI_failure_type` (known technical failures from GMF classification)

4. **Compute Impact Scores**
   - Each incident is scored based on impact metrics:
     - AI Harm Level
     - Lives lost
     - Injuries
     - Number of entities affected
   - A weighted formula prioritizes incidents that **actually caused significant harm** rather than just having lengthy descriptions or media coverage.

5. **Select Top Incidents**
   - Incidents are ranked by the computed impact score.
   - The **top 50 incidents** are retained for downstream analysis.

6. **Output**
   - Generates a CSV `malicious_top50_incidents.csv` with the following fields:
     ```
     incident_id, title, description, harm_category, entities, AI_system, AI_task, AI_failure_type
     ```
   - This dataset is ready for **semantic clustering, use-case analysis, and mitigation strategy development**.

> ⚡ This notebook ensures that the selected incidents represent **real AI risks** and technical failures, rather than being driven by media attention or description length alone.


In [13]:
import pandas as pd
import json
import numpy as np

# --- Step 1: Load CSV and JSON ---
classified_df = pd.read_csv('/content/llm_classified_incidents.csv')
with open('/content/unified_incidents.json', 'r') as f:
    full_incidents = json.load(f)

# --- Step 2: Filter CSV for Malicious Use & Security ---
malicious_df = classified_df[classified_df['harm_category'] == "Malicious Use & Security"]

# --- Step 3: Build lookup dict from JSON ---
incident_lookup = {str(inc['incident_id']): inc for inc in full_incidents}

# Convert entities list to a readable string
def clean_entities(entity_list):
    if isinstance(entity_list, list):
        # Join only string elements, ignore junk
        clean_list = [str(e) for e in entity_list if isinstance(e, str) and e.strip()]
        return ", ".join(clean_list) if clean_list else None
    return None

# --- Step 4: Collect incidents with scoring ---
selected_incidents = []
for idx, row in malicious_df.iterrows():
    inc_id = str(row['incident_id'])
    if inc_id in incident_lookup:
        inc = incident_lookup[inc_id]
        entities = inc.get('entities', [])

        # Classification info
        classifications = inc.get('classifications', {})
        csetv1 = classifications.get('classifications_CSETv1', {})
        AI_system = csetv1.get('AI System', 0)
        AI_task = csetv1.get('AI Task', None)
        AI_failure_type = classifications.get('classifications_GMF', {}).get('Known AI Technical Failure', None)

        # Impact metrics
        try:
            AI_harm_level = float(csetv1.get('AI Harm Level', 0))  # scale 0-5
        except:
            AI_harm_level = 0
        try:
            lives_lost = float(inc.get('Lives Lost', 0))
        except:
            lives_lost = 0
        try:
            injuries = float(inc.get('Injuries', 0))
        except:
            injuries = 0
        # Optional: Number of entities impacted
        entities_count = len(entities) if entities else 0

        # Compute weighted score
        score = (AI_harm_level * 3) + (lives_lost * 5) + (injuries * 2) + (entities_count * 1)

        selected_incidents.append({
            'incident_id': row['incident_id'],
            'title': row['title'],
            'description': row['description'],
            'harm_category': row['harm_category'],
            'entities': clean_entities(entities),
            'AI_system': AI_system,
            'AI_task': AI_task,
            'AI_failure_type': AI_failure_type,
            'impact_score': score
        })

# --- Step 5: Sort by impact_score and pick top 50 ---
selected_incidents = sorted(selected_incidents, key=lambda x: x['impact_score'], reverse=True)[:50]

# --- Step 6: Convert to DataFrame and save CSV ---
output_df = pd.DataFrame(selected_incidents).drop(columns=['impact_score'])
output_df.to_csv('/content/malicious_top50_incidents.csv', index=False)

# Optional: Display top incidents
output_df.head(10)


,incident_id,title,description,harm_category,entities,AI_system,AI_task,AI_failure_type
0,286,"TikTok’s ""For You"" Allegedly Pushed Fatal “Bla...",TikTok’s recommendation algorithm was alleged ...,Malicious Use & Security,None,0,None,"Unsafe Exposure or Access, Lack of Safety Prot..."
1,1195,Nigeria-Based YouTube Network Allegedly Uses A...,A network of five Nigeria-based YouTube channe...,Malicious Use & Security,None,0,None,None
2,302,Students Allegedly Wrongfully Accused of Cheat...,Dartmouth's Geisel School of Medicine allegedl...,Malicious Use & Security,None,0,None,None
3,1189,Joann Fabrics Shoppers Reportedly Defrauded by...,Consumers were allegedly defrauded by AI-gener...,Malicious Use & Security,None,0,None,None
4,1182,Purportedly AI-Generated Video of Tigers at Ba...,A purportedly AI-generated video circulated on...,Malicious Use & Security,None,0,None,None
5,944,Kenya's Foreign Affairs Principal Secretary Ko...,Kenya's Foreign Affairs Principal Secretary (P...,Malicious Use & Security,None,0,None,None
6,953,Deepfake Videos of Barbara O’Neill Allegedly U...,"Deepfake videos of Barbara O’Neill, an Austral...",Malicious Use & Security,None,0,None,None
7,567,Deepfake Voice Exploit Compromises Retool's Cl...,"In August 2023, a hacker reportedly was succes...",Malicious Use & Security,None,0,None,None
8,614,Google Bard Allegedly Generates False Allegati...,Australian academics reportedly used Google Ba...,Malicious Use & Security,None,yes,"chatbot, content generation",None
9,745,Figma Disables AI Feature After Accusations of...,Figma has temporarily disabled its AI design f...,Malicious Use & Security,None,0,None,None
